In [1]:
# ─── Cell 1: Install dependencies ───────────────────────────────────────────
!pip install llama-index-core llama-index-llms-groq llama-index-embeddings-huggingface transformers torch sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   ------ --------------------------------- 1.8/12.0 MB 10.7 MB/s eta 0:00:01
   ------------- -------------------------- 3.9/12.0 MB 10.3 MB/s eta 0:00:01
   -------------------- ------------------- 6.3/12.0 MB 10.5 MB/s eta 0:00:01
   ---------------------------- ----------- 8.7/12.0 MB 10.6 MB/s eta 0:00:01
   ----------------------------------- ---- 10.7/12.0 MB 10.7 MB/s eta 0:00:01
   ---------------------------------------- 12.0/12.0 MB 10.4 MB/s  0:00:01
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
   ---------------------------------------- 0.0/588.9 kB ? eta -:--:--
   ---------------------------------------- 588.9/588.9 kB 9.5 MB/s  0:00:00

  Attempting uninstall: huggingface-hub

    Found existing installation: huggingface_hub 1.4.1

    U

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-index 0.14.22 requires llama-index-llms-openai<0.8,>=0.7.0, but you have llama-index-llms-openai 0.6.26 which is incompatible.


In [2]:
# ─── Cell 2: Imports ─────────────────────────────────────────────────────────
import os
import json
from llama_index.core import (
    VectorStoreIndex,
    Document,
    Settings,
    StorageContext,
    load_index_from_storage,
)
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [ ]:
# ─── Cell 3: Configuration ───────────────────────────────────────────────────
os.environ["GROQ_API_KEY"] = ""

Settings.llm = Groq(model="llama-3.1-8b-instant", temperature=0)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

STORAGE_DIR = "C:/Users/USER/Downloads/silicon_task/rag/llamaindex_rag/storage"
DATA_PATH   = "C:/Users/USER/Downloads/silicon_task/rag/llamaindex_rag/data/Aircraft_Type_Designators.json"

# ─── Cell 4: Load & parse JSON into Documents ────────────────────────────────
with open(DATA_PATH, "r") as f:
    raw_data = json.load(f)

documents = []
for aircraft_code, triples_text in raw_data.items():
    lines    = [l.strip() for l in triples_text.strip().split("\n") if l.strip()]
    metadata = {"aircraft_code": aircraft_code, "triple_count": len(lines)}

    doc_text  = f"Aircraft Code: {aircraft_code}\n\n"
    doc_text += "Knowledge Graph Triples:\n"
    doc_text += "\n".join(lines)

    documents.append(Document(text=doc_text, metadata=metadata))

print(f"Loaded {len(documents)} documents")
print("\nSample document:\n", documents[0].text[:300])

# ─── Cell 5: Build or load the VectorStore index ─────────────────────────────
if os.path.exists(STORAGE_DIR) and os.listdir(STORAGE_DIR):
    print("Loading existing index from storage...")
    storage_context = StorageContext.from_defaults(persist_dir=STORAGE_DIR)
    index = load_index_from_storage(storage_context)
else:
    print("Building new index...")
    index = VectorStoreIndex.from_documents(documents, show_progress=True)
    index.storage_context.persist(persist_dir=STORAGE_DIR)
    print(f"Index saved to {STORAGE_DIR}")

# ─── Cell 6: Create query engine ─────────────────────────────────────────────
query_engine = index.as_query_engine(
    similarity_top_k=3,
    response_mode="compact",
)

# ─── Cell 7: Query examples ──────────────────────────────────────────────────
queries = [
    "Who manufactures the J328 aircraft?",
    "What engine type does UL45 use?",
    "Which aircraft has WTC category M?",
    "List all LandPlane aircraft models.",
]

for q in queries:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    response = query_engine.query(q)
    print(f"A: {response}")

# ─── Cell 8: Interactive query loop ──────────────────────────────────────────
while True:
    user_query = input("\nEnter your question (or 'quit' to exit): ").strip()
    if user_query.lower() in ("quit", "exit", "q"):
        break
    if user_query:
        response = query_engine.query(user_query)
        print(f"\nAnswer: {response}")
        if hasattr(response, "source_nodes"):
            print("\nSources:")
            for node in response.source_nodes:
                code  = node.metadata.get("aircraft_code", "unknown")
                score = round(node.score, 4) if node.score else "N/A"
                print(f"  - Aircraft {code} (similarity: {score})")

Loaded 219 documents

Sample document:
 Aircraft Code: J328

Knowledge Graph Triples:
(J328, isManufacturedBy, 328 SUPPORT SERVICES)
(J328, hasModel, Dornier 328JET)
(J328, isTypeOf, LandPlane)
(J328, hasEngineType, Jet)
(J328, hasEngineCount, 2)
(J328, hasWTC, M)
(J328, isManufacturedBy, AVCRAFT)
(J328, hasModel, Dornier 328JET)
(J328, i
Building new index...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/240 [00:00<?, ?it/s]

Index saved to C:/Users/USER/Downloads/silicon_task/rag/llamaindex_rag/storage

Q: Who manufactures the J328 aircraft?
A: FAIRCHILD DORNIER and RUAG manufacture the J328 aircraft.

Q: What engine type does UL45 use?
A: The aircraft type of UL45 uses Piston engines.

Q: Which aircraft has WTC category M?
A: There is no aircraft with WTC category M in the provided information.

Q: List all LandPlane aircraft models.
A: L-29 Delfin, A-700 AdamJet, 90, M-10 Cadet
